# Advanced Problems with Solutions: Iterating Collections in Python

This notebook extends the topic of **iterating collections** into advanced practice.

You will build several custom iterables and iterators, compare one-shot and reusable iteration designs, implement generator-based tools, and test edge cases.

## Learning goals

By the end, you should be able to:

1. Explain the difference between an **iterable** and an **iterator**.
2. Implement `__iter__()` and `__next__()` correctly.
3. Use `StopIteration` correctly.
4. Decide when an object should be a one-shot iterator vs. a reusable iterable collection.
5. Write lazy generator functions that compose cleanly.
6. Test iteration behavior, exhaustion, nested loops, and edge cases.


## Setup

Run this cell first. It imports tools used throughout the notebook.


In [1]:
from __future__ import annotations

from collections import deque
from collections.abc import Iterable, Iterator
from dataclasses import dataclass
from itertools import islice
import random
from typing import Any, Callable, Optional


## Best-practice checklist

When designing custom iteration:

- Use `iter(obj)` to obtain an iterator.
- Use `next(iterator)` to request the next item.
- Raise `StopIteration` only when iteration is exhausted.
- If the object is its own iterator, `__iter__` usually returns `self`; this creates a **one-shot** iterator.
- If the object should be reusable, make `__iter__` return a **fresh iterator object** every time.
- Do not store reusable iteration state on the collection itself unless you intentionally want one-shot behavior.
- Prefer generators for simple iteration logic.
- Prefer separate iterator classes for complex state, validation, or reusable container APIs.


## Helper: inspect iterable vs. iterator behavior

This small helper makes the protocol visible.


In [2]:
def describe_iteration_object(obj: Any) -> None:
    print(f"object: {obj!r}")
    print(f"type: {type(obj).__name__}")
    print(f"is Iterable: {isinstance(obj, Iterable)}")
    print(f"is Iterator: {isinstance(obj, Iterator)}")
    try:
        iterator = iter(obj)
    except TypeError as ex:
        print(f"iter(obj): TypeError -> {ex}")
    else:
        print(f"iter(obj) type: {type(iterator).__name__}")
        print(f"iter(obj) is obj: {iterator is obj}")


# Concept examples before the problems

These examples build intuition before the challenge section.


## Example A: Index-only objects can still be iterated by Python

If an object implements `__getitem__`, Python can repeatedly request indices `0`, `1`, `2`, ... until `IndexError` is raised.

This is an older sequence-style iteration fallback. It works, but it only fits indexable sequences.


In [3]:
class IndexOnlySquares:
    def __init__(self, length: int):
        self.length = length

    def __getitem__(self, index: int) -> int:
        if not isinstance(index, int):
            raise TypeError("index must be an integer")
        if index < 0 or index >= self.length:
            raise IndexError(index)
        return index ** 2

seq = IndexOnlySquares(5)
print(list(seq))
describe_iteration_object(seq)


[0, 1, 4, 9, 16]
object: <__main__.IndexOnlySquares object at 0x0000021FEB908440>
type: IndexOnlySquares
is Iterable: False
is Iterator: False
iter(obj) type: iterator
iter(obj) is obj: False


## Example B: A `__next__` method alone is not enough for `for` loops

`next(obj)` can work, but `for item in obj` still requires `iter(obj)` to succeed.


In [4]:
class NextOnlySquares:
    def __init__(self, length: int):
        self.length = length
        self.index = 0

    def __next__(self) -> int:
        if self.index >= self.length:
            raise StopIteration
        value = self.index ** 2
        self.index += 1
        return value

x = NextOnlySquares(3)
print(next(x))
print(next(x))
describe_iteration_object(x)

try:
    for item in x:
        print(item)
except TypeError as ex:
    print("for-loop failed:", ex)


0
1
object: <__main__.NextOnlySquares object at 0x0000021FFB8CE7B0>
type: NextOnlySquares
is Iterable: False
is Iterator: False
iter(obj): TypeError -> 'NextOnlySquares' object is not iterable
for-loop failed: 'NextOnlySquares' object is not iterable


## Example C: A true iterator implements both `__iter__` and `__next__`

For iterator objects, `__iter__` normally returns `self`.


In [5]:
class OneShotSquares:
    def __init__(self, length: int):
        self.length = length
        self.index = 0

    def __iter__(self) -> "OneShotSquares":
        return self

    def __next__(self) -> int:
        if self.index >= self.length:
            raise StopIteration
        value = self.index ** 2
        self.index += 1
        return value

it = OneShotSquares(5)
print(list(it))
print(list(it))  # already exhausted

describe_iteration_object(OneShotSquares(2))


[0, 1, 4, 9, 16]
[]
object: <__main__.OneShotSquares object at 0x0000021FEB750A50>
type: OneShotSquares
is Iterable: True
is Iterator: True
iter(obj) type: OneShotSquares
iter(obj) is obj: True


# Advanced problems with solutions

Each problem includes:

- requirements,
- a complete solution,
- demonstration code,
- and assertions to verify behavior.


## Problem 1 — Build a robust one-shot squares iterator

Create a class `SquaresIterator` with the following behavior:

1. `SquaresIterator(n)` produces `0 ** 2`, `1 ** 2`, ..., `(n - 1) ** 2`.
2. It works with `next(obj)`.
3. It works in a `for` loop.
4. It raises `StopIteration` when exhausted.
5. It validates that `n` is a non-negative integer.
6. It is intentionally one-shot: after exhaustion, a second loop produces no values.


In [6]:
class SquaresIterator:
    def __init__(self, n: int):
        if not isinstance(n, int):
            raise TypeError("n must be an integer")
        if n < 0:
            raise ValueError("n must be non-negative")
        self.n = n
        self.current = 0

    def __iter__(self) -> "SquaresIterator":
        return self

    def __next__(self) -> int:
        if self.current >= self.n:
            raise StopIteration
        value = self.current ** 2
        self.current += 1
        return value

# Demonstration
sq = SquaresIterator(5)
print(next(sq))
print(next(sq))
print(list(sq))      # remaining values only
print(list(sq))      # exhausted

# Tests
assert list(SquaresIterator(0)) == []
assert list(SquaresIterator(1)) == [0]
assert list(SquaresIterator(5)) == [0, 1, 4, 9, 16]

try:
    SquaresIterator(-1)
except ValueError:
    pass
else:
    raise AssertionError("negative n should fail")

try:
    SquaresIterator(3.5)
except TypeError:
    pass
else:
    raise AssertionError("non-integer n should fail")


0
1
[4, 9, 16]
[]


### Key takeaway from Problem 1

`SquaresIterator` is both an iterable and an iterator, but it is **not reusable**. Its internal cursor moves forward forever.


## Problem 2 — Build a reusable squares collection using a separate iterator

Create a reusable `Squares` collection.

Requirements:

1. `list(Squares(5))` returns `[0, 1, 4, 9, 16]`.
2. Running `list()` multiple times gives the same result.
3. Nested loops work correctly.
4. `len(Squares(5)) == 5`.
5. Iteration state must live in a separate iterator object, not on the collection.


In [7]:
class SquaresCursor:
    def __init__(self, n: int):
        self.n = n
        self.current = 0

    def __iter__(self) -> "SquaresCursor":
        return self

    def __next__(self) -> int:
        if self.current >= self.n:
            raise StopIteration
        value = self.current ** 2
        self.current += 1
        return value

class Squares:
    def __init__(self, n: int):
        if not isinstance(n, int):
            raise TypeError("n must be an integer")
        if n < 0:
            raise ValueError("n must be non-negative")
        self.n = n

    def __len__(self) -> int:
        return self.n

    def __iter__(self) -> SquaresCursor:
        return SquaresCursor(self.n)

sq = Squares(4)
print(list(sq))
print(list(sq))
print(len(sq))

pairs = [(a, b) for a in Squares(3) for b in Squares(3)]
print(pairs)

assert list(Squares(4)) == [0, 1, 4, 9]
assert list(Squares(4)) == list(Squares(4))
assert len(Squares(10)) == 10
assert len(pairs) == 9
assert pairs[0] == (0, 0)
assert pairs[-1] == (4, 4)


[0, 1, 4, 9]
[0, 1, 4, 9]
4
[(0, 0), (0, 1), (0, 4), (1, 0), (1, 1), (1, 4), (4, 0), (4, 1), (4, 4)]


### Key takeaway from Problem 2

Reusable collections should usually return a **new iterator** every time `iter(collection)` is called.


## Problem 3 — Build a lazy square sequence with indexing and slicing

Create `SquareSequence`, a memory-efficient object that behaves like a sequence without storing every square.

Requirements:

1. `SquareSequence(6)[3] == 9`.
2. Negative indices work: `SquareSequence(6)[-1] == 25`.
3. Slices return lists: `SquareSequence(6)[1:5:2] == [1, 9]`.
4. It is reusable in loops.
5. It validates bounds and raises `IndexError` for invalid integer indexes.


In [8]:
class SquareSequence:
    def __init__(self, n: int):
        if not isinstance(n, int):
            raise TypeError("n must be an integer")
        if n < 0:
            raise ValueError("n must be non-negative")
        self.n = n

    def __len__(self) -> int:
        return self.n

    def __getitem__(self, index):
        if isinstance(index, slice):
            return [i ** 2 for i in range(*index.indices(self.n))]

        if not isinstance(index, int):
            raise TypeError("index must be an integer or slice")

        if index < 0:
            index += self.n

        if index < 0 or index >= self.n:
            raise IndexError(index)

        return index ** 2

    def __iter__(self):
        for i in range(self.n):
            yield i ** 2

seq = SquareSequence(6)
print(seq[3])
print(seq[-1])
print(seq[1:5:2])
print(list(seq))
print(list(seq))

assert seq[3] == 9
assert seq[-1] == 25
assert seq[1:5:2] == [1, 9]
assert list(seq) == [0, 1, 4, 9, 16, 25]

try:
    seq[100]
except IndexError:
    pass
else:
    raise AssertionError("out-of-bounds access should fail")


9
25
[1, 9]
[0, 1, 4, 9, 16, 25]
[0, 1, 4, 9, 16, 25]


### Key takeaway from Problem 3

Iteration and indexing are related, but they are not the same protocol. A class can support both when it makes sense.


## Problem 4 — Make random-number iteration deterministic and reusable

The original `RandomNumbers` idea generates values on demand. Improve it using best practices.

Requirements:

1. `RandomNumbers(n, low, high, seed)` is reusable.
2. Each iteration produces the same values for the same seed.
3. Different seeds usually produce different values.
4. Do not use the global random generator directly.
5. `low` and `high` are inclusive.


In [9]:
class RandomNumberIterator:
    def __init__(self, n: int, low: int, high: int, seed: Optional[int]):
        self.remaining = n
        self.low = low
        self.high = high
        self.rng = random.Random(seed)

    def __iter__(self) -> "RandomNumberIterator":
        return self

    def __next__(self) -> int:
        if self.remaining <= 0:
            raise StopIteration
        self.remaining -= 1
        return self.rng.randint(self.low, self.high)

class RandomNumbers:
    def __init__(self, n: int, low: int = 0, high: int = 10, seed: Optional[int] = None):
        if not isinstance(n, int):
            raise TypeError("n must be an integer")
        if n < 0:
            raise ValueError("n must be non-negative")
        if low > high:
            raise ValueError("low must be <= high")
        self.n = n
        self.low = low
        self.high = high
        self.seed = seed

    def __len__(self) -> int:
        return self.n

    def __iter__(self) -> RandomNumberIterator:
        return RandomNumberIterator(self.n, self.low, self.high, self.seed)

nums = RandomNumbers(8, 1, 6, seed=42)
first_pass = list(nums)
second_pass = list(nums)
print(first_pass)
print(second_pass)

assert first_pass == second_pass
assert len(first_pass) == 8
assert all(1 <= x <= 6 for x in first_pass)
assert list(RandomNumbers(5, seed=1)) != list(RandomNumbers(5, seed=2))


[6, 1, 1, 6, 3, 2, 2, 2]
[6, 1, 1, 6, 3, 2, 2, 2]


### Key takeaway from Problem 4

A reusable iterable that generates random values should create a fresh random generator inside each iterator, especially when deterministic reproducibility matters.


## Problem 5 — Build an infinite arithmetic progression iterator

Create `ArithmeticProgression`.

Requirements:

1. It starts at `start`.
2. Each call adds `step`.
3. It is infinite.
4. It works with `next()` and `itertools.islice()`.
5. It should not be converted to `list()` without slicing.


In [10]:
class ArithmeticProgression:
    def __init__(self, start: int = 0, step: int = 1):
        self.current = start
        self.step = step

    def __iter__(self) -> "ArithmeticProgression":
        return self

    def __next__(self) -> int:
        value = self.current
        self.current += self.step
        return value

ap = ArithmeticProgression(start=10, step=3)
print(next(ap))
print(next(ap))
print(list(islice(ap, 5)))

assert list(islice(ArithmeticProgression(0, 2), 6)) == [0, 2, 4, 6, 8, 10]
assert list(islice(ArithmeticProgression(5, -1), 5)) == [5, 4, 3, 2, 1]


10
13
[16, 19, 22, 25, 28]


### Key takeaway from Problem 5

Infinite iterators are powerful, but they must be consumed with limiting tools such as `islice`, `take`, or a `break` condition.


## Problem 6 — Rewrite a class-based iterator as a generator

Implement `triangular_numbers(n)`.

The triangular sequence is:

```text
1, 3, 6, 10, 15, ...
```

Requirements:

1. Use `yield`.
2. Generate exactly `n` values.
3. Validate that `n` is a non-negative integer.
4. Keep the implementation lazy.


In [11]:
def triangular_numbers(n: int):
    if not isinstance(n, int):
        raise TypeError("n must be an integer")
    if n < 0:
        raise ValueError("n must be non-negative")

    total = 0
    for i in range(1, n + 1):
        total += i
        yield total

print(list(triangular_numbers(8)))

assert list(triangular_numbers(0)) == []
assert list(triangular_numbers(1)) == [1]
assert list(triangular_numbers(5)) == [1, 3, 6, 10, 15]

g = triangular_numbers(3)
assert iter(g) is g
assert next(g) == 1
assert next(g) == 3
assert next(g) == 6
try:
    next(g)
except StopIteration:
    pass
else:
    raise AssertionError("generator should be exhausted")


[1, 3, 6, 10, 15, 21, 28, 36]


### Key takeaway from Problem 6

Generator functions are often the cleanest way to create iterators when the iteration state is simple.


## Problem 7 — Implement lazy `take`, `drop`, and `first_where`

Create three utility functions:

1. `take(iterable, n)` returns a list of at most `n` items.
2. `drop(iterable, n)` skips `n` items and returns the remaining iterator.
3. `first_where(iterable, predicate, default=None)` returns the first matching value or `default`.

Requirements:

- They must work with lists, generators, and custom iterators.
- They should not consume more values than necessary.


In [12]:
def take(iterable: Iterable, n: int) -> list:
    if n < 0:
        raise ValueError("n must be non-negative")
    iterator = iter(iterable)
    result = []
    for _ in range(n):
        try:
            result.append(next(iterator))
        except StopIteration:
            break
    return result


def drop(iterable: Iterable, n: int) -> Iterator:
    if n < 0:
        raise ValueError("n must be non-negative")
    iterator = iter(iterable)
    for _ in range(n):
        try:
            next(iterator)
        except StopIteration:
            break
    return iterator


def first_where(iterable: Iterable, predicate: Callable[[Any], bool], default=None):
    for item in iterable:
        if predicate(item):
            return item
    return default

numbers = ArithmeticProgression(0, 5)
print(take(numbers, 4))
print(next(numbers))  # confirms only 4 values were consumed

remaining = drop([10, 20, 30, 40, 50], 2)
print(list(remaining))

print(first_where(range(20), lambda x: x > 0 and x % 7 == 0))
print(first_where(range(5), lambda x: x > 10, default="not found"))

assert take([1, 2, 3], 10) == [1, 2, 3]
assert take(ArithmeticProgression(1, 1), 5) == [1, 2, 3, 4, 5]
assert list(drop([1, 2, 3, 4], 2)) == [3, 4]
assert first_where([1, 3, 8, 9], lambda x: x % 2 == 0) == 8
assert first_where([1, 3, 5], lambda x: x % 2 == 0, default=None) is None


[0, 5, 10, 15]
20
[30, 40, 50]
7
not found


### Key takeaway from Problem 7

Iterator utilities should consume only as much input as needed. This makes them safe for large or infinite streams.


## Problem 8 — Implement a `Peekable` iterator wrapper

Create a wrapper that lets you preview the next item without consuming it.

Requirements:

1. `peek()` returns the next value without advancing the iterator.
2. Repeated `peek()` calls return the same item.
3. `next()` consumes the peeked item.
4. `peek(default)` returns `default` instead of raising if the iterator is empty.
5. `next()` still raises `StopIteration` when empty.


In [13]:
_MISSING = object()

class Peekable:
    def __init__(self, iterable: Iterable):
        self._iterator = iter(iterable)
        self._cache = deque()

    def __iter__(self) -> "Peekable":
        return self

    def __next__(self):
        if self._cache:
            return self._cache.popleft()
        return next(self._iterator)

    def peek(self, default=_MISSING):
        if not self._cache:
            try:
                self._cache.append(next(self._iterator))
            except StopIteration:
                if default is _MISSING:
                    raise
                return default
        return self._cache[0]

p = Peekable([10, 20, 30])
print(p.peek())
print(p.peek())
print(next(p))
print(next(p))
print(list(p))
print(p.peek("EMPTY"))

p = Peekable(iter([1, 2]))
assert p.peek() == 1
assert p.peek() == 1
assert next(p) == 1
assert p.peek() == 2
assert next(p) == 2
assert p.peek(default="done") == "done"
try:
    next(p)
except StopIteration:
    pass
else:
    raise AssertionError("empty Peekable should raise StopIteration on next")


10
10
10
20
[30]
EMPTY


### Key takeaway from Problem 8

A small cache is enough to implement lookahead while preserving normal iterator behavior.


## Problem 9 — Implement lazy batching

Write `batched(iterable, size, strict=False)`.

Requirements:

1. Yield tuples of length `size`.
2. The final tuple may be shorter if `strict=False`.
3. If `strict=True`, raise `ValueError` when the final batch is incomplete.
4. Do not materialize the entire input.
5. Validate that `size >= 1`.


In [14]:
def batched(iterable: Iterable, size: int, *, strict: bool = False):
    if not isinstance(size, int):
        raise TypeError("size must be an integer")
    if size < 1:
        raise ValueError("size must be at least 1")

    iterator = iter(iterable)
    while True:
        batch = tuple(islice(iterator, size))
        if not batch:
            return
        if strict and len(batch) != size:
            raise ValueError("incomplete final batch")
        yield batch

print(list(batched(range(10), 3)))
print(list(batched("ABCDEFG", 2)))
print(take(batched(ArithmeticProgression(1, 1), 4), 3))

assert list(batched(range(7), 3)) == [(0, 1, 2), (3, 4, 5), (6,)]
assert list(batched(range(6), 3, strict=True)) == [(0, 1, 2), (3, 4, 5)]
try:
    list(batched(range(7), 3, strict=True))
except ValueError:
    pass
else:
    raise AssertionError("strict batching should reject incomplete final batch")


[(0, 1, 2), (3, 4, 5), (6, 7, 8), (9,)]
[('A', 'B'), ('C', 'D'), ('E', 'F'), ('G',)]
[(1, 2, 3, 4), (5, 6, 7, 8), (9, 10, 11, 12)]


### Key takeaway from Problem 9

Batching should be lazy: at any moment, it only needs to hold one batch in memory.


## Problem 10 — Implement recursive flattening safely

Write `flatten(items)`.

Requirements:

1. Recursively flatten nested iterables.
2. Do not split strings or bytes into characters.
3. Work lazily using `yield` / `yield from`.
4. Support arbitrary nesting of lists, tuples, generators, and ranges.


In [15]:
def flatten(items: Iterable):
    for item in items:
        if isinstance(item, Iterable) and not isinstance(item, (str, bytes)):
            yield from flatten(item)
        else:
            yield item

nested = [1, [2, 3], (4, [5, range(6, 9)]), "abc", b"xy"]
print(list(flatten(nested)))

assert list(flatten([1, [2, [3, 4]], 5])) == [1, 2, 3, 4, 5]
assert list(flatten(["abc", ["de", "f"]])) == ["abc", "de", "f"]
assert list(flatten([range(3), (x for x in [3, 4])])) == [0, 1, 2, 3, 4]


[1, 2, 3, 4, 5, 6, 7, 8, 'abc', b'xy']


### Key takeaway from Problem 10

Recursive iteration is powerful, but strings are also iterable. Most flattening utilities should treat strings as atomic values.


## Problem 11 — Deduplicate lazily while preserving order

Write `unique_everseen(iterable, key=None)`.

Requirements:

1. Yield each unique item once.
2. Preserve the first-seen order.
3. Accept an optional `key` function.
4. Work lazily.


In [16]:
def unique_everseen(iterable: Iterable, key: Optional[Callable[[Any], Any]] = None):
    seen = set()
    for item in iterable:
        marker = item if key is None else key(item)
        if marker not in seen:
            seen.add(marker)
            yield item

print(list(unique_everseen([1, 2, 1, 3, 2, 4, 1])))
print(list(unique_everseen(["Apple", "apple", "BANANA", "banana"], key=str.lower)))

assert list(unique_everseen([1, 1, 2, 3, 2])) == [1, 2, 3]
assert list(unique_everseen(["A", "a", "B"], key=str.lower)) == ["A", "B"]

stream = ArithmeticProgression(0, 1)
first_five_unique_parity = take(unique_everseen(stream, key=lambda x: x % 2), 2)
assert first_five_unique_parity == [0, 1]


[1, 2, 3, 4]
['Apple', 'BANANA']


### Key takeaway from Problem 11

Laziness and state can coexist. Here, the function only stores keys it has already emitted.


## Problem 12 — Recreate `enumerate` lazily

Implement `my_enumerate(iterable, start=0)`.

Requirements:

1. Yield `(index, item)` pairs.
2. Support any iterable.
3. Do not convert the iterable to a list.
4. Support custom start values.


In [17]:
def my_enumerate(iterable: Iterable, start: int = 0):
    index = start
    for item in iterable:
        yield index, item
        index += 1

print(list(my_enumerate(["a", "b", "c"])))
print(list(my_enumerate("xyz", start=10)))
print(take(my_enumerate(ArithmeticProgression(100, 10), start=1), 4))

assert list(my_enumerate(["a", "b"])) == [(0, "a"), (1, "b")]
assert list(my_enumerate(["a", "b"], start=5)) == [(5, "a"), (6, "b")]


[(0, 'a'), (1, 'b'), (2, 'c')]
[(10, 'x'), (11, 'y'), (12, 'z')]
[(1, 100), (2, 110), (3, 120), (4, 130)]


### Key takeaway from Problem 12

Many built-in iteration tools are simple protocols plus careful lazy behavior.


## Problem 13 — Recreate a simplified `zip` lazily

Implement `my_zip(*iterables)`.

Requirements:

1. Yield tuples of one item from each iterable.
2. Stop when the shortest iterable is exhausted.
3. Work with infinite iterators if at least one finite iterable limits the output.
4. Do not materialize the inputs.


In [21]:
from collections.abc import Iterable


def my_zip(*iterables: Iterable):
    if not iterables:
        return

    iterators = [iter(iterable) for iterable in iterables]

    while True:
        row = []

        for iterator in iterators:
            try:
                row.append(next(iterator))
            except StopIteration:
                return

        yield tuple(row)


print(list(my_zip([1, 2, 3], "abc")))
print(list(my_zip([1, 2], "abcdef")))
print(list(my_zip(range(3), ArithmeticProgression(10, 10))))

assert list(my_zip([1, 2], ["a", "b", "c"])) == [(1, "a"), (2, "b")]
assert list(my_zip()) == []
assert list(my_zip(range(3), ArithmeticProgression(0, 5))) == [(0, 0), (1, 5), (2, 10)]


[(1, 'a'), (2, 'b'), (3, 'c')]
[(1, 'a'), (2, 'b')]
[(0, 10), (1, 20), (2, 30)]


### Key takeaway from Problem 13

A zip-like iterator must stop at the first exhausted input.


## Problem 14 — Stream-process log lines with an iterator pipeline

Build a small lazy pipeline for log processing.

Input lines look like this:

```text
2026-06-19 INFO START service=api
2026-06-19 ERROR E42 service=db
```

Requirements:

1. Parse each line into a `LogEntry` dataclass.
2. Skip blank lines.
3. Ignore malformed lines rather than crashing.
4. Lazily yield only error entries.
5. Count error codes without first building a huge list.


In [22]:
@dataclass(frozen=True)
class LogEntry:
    date: str
    level: str
    message: str
    metadata: dict[str, str]


def parse_log_lines(lines: Iterable[str]):
    for raw_line in lines:
        line = raw_line.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) < 3:
            continue

        date, level, message, *metadata_parts = parts
        metadata = {}
        malformed = False
        for part in metadata_parts:
            if "=" not in part:
                malformed = True
                break
            key, value = part.split("=", 1)
            metadata[key] = value

        if malformed:
            continue

        yield LogEntry(date=date, level=level, message=message, metadata=metadata)


def only_errors(entries: Iterable[LogEntry]):
    for entry in entries:
        if entry.level == "ERROR":
            yield entry


def count_error_codes(entries: Iterable[LogEntry]) -> dict[str, int]:
    counts = {}
    for entry in entries:
        counts[entry.message] = counts.get(entry.message, 0) + 1
    return counts

lines = [
    "2026-06-19 INFO START service=api",
    "2026-06-19 ERROR E42 service=db",
    "",
    "bad line",
    "2026-06-19 ERROR E42 service=db retry=true",
    "2026-06-19 ERROR E99 service=cache",
    "2026-06-19 WARN SLOW service=api ms=900",
]

entries = parse_log_lines(lines)
errors = only_errors(entries)
counts = count_error_codes(errors)
print(counts)

assert counts == {"E42": 2, "E99": 1}
assert list(only_errors(parse_log_lines(["2026-06-19 INFO OK service=api"]))) == []


{'E42': 2, 'E99': 1}


### Key takeaway from Problem 14

Iterator pipelines let you process streams step by step without loading all intermediate results into memory.


## Problem 15 — Build a resettable iterator carefully

Most Python iterators do not reset themselves. But sometimes you may want an object that can be manually reset.

Create `ResettableCountdown`.

Requirements:

1. `ResettableCountdown(3)` yields `3, 2, 1`.
2. `reset()` restarts the iterator.
3. `for` loops work.
4. Exhaustion raises `StopIteration`.
5. Validation rejects negative starts.


In [23]:
class ResettableCountdown:
    def __init__(self, start: int):
        if not isinstance(start, int):
            raise TypeError("start must be an integer")
        if start < 0:
            raise ValueError("start must be non-negative")
        self.start = start
        self.current = start

    def __iter__(self) -> "ResettableCountdown":
        return self

    def __next__(self) -> int:
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value

    def reset(self) -> None:
        self.current = self.start

countdown = ResettableCountdown(3)
print(list(countdown))
print(list(countdown))
countdown.reset()
print(list(countdown))

assert list(ResettableCountdown(3)) == [3, 2, 1]
cd = ResettableCountdown(2)
assert next(cd) == 2
assert next(cd) == 1
try:
    next(cd)
except StopIteration:
    pass
else:
    raise AssertionError("countdown should be exhausted")
cd.reset()
assert list(cd) == [2, 1]


[3, 2, 1]
[]
[3, 2, 1]


### Key takeaway from Problem 15

Manual reset can be useful, but it is not the default Python iterator pattern. Reusable collections are usually clearer than resettable iterators.


# Extra challenge prompts

Try these without looking back at the solutions above:

1. Modify `Peekable` to support `prepend(value)` so a value is returned before the rest of the stream.
2. Modify `batched` to yield lists instead of tuples.
3. Create `sliding_window(iterable, size)` that yields overlapping windows.
4. Create `round_robin(*iterables)` that alternates values from multiple iterables until all are exhausted.
5. Create `consume_until(iterable, predicate)` that yields items until the predicate becomes true.


## Extra Challenge Solution 1 — `PrependablePeekable`


In [24]:
class PrependablePeekable(Peekable):
    def prepend(self, value) -> None:
        self._cache.appendleft(value)

p = PrependablePeekable([2, 3])
p.prepend(1)
print(list(p))

assert list(PrependablePeekable([])) == []
p = PrependablePeekable(["b", "c"])
p.prepend("a")
assert list(p) == ["a", "b", "c"]


[1, 2, 3]


## Extra Challenge Solution 2 — `sliding_window`


In [25]:
def sliding_window(iterable: Iterable, size: int):
    if not isinstance(size, int):
        raise TypeError("size must be an integer")
    if size < 1:
        raise ValueError("size must be at least 1")

    iterator = iter(iterable)
    window = deque(islice(iterator, size), maxlen=size)
    if len(window) < size:
        return

    yield tuple(window)
    for item in iterator:
        window.append(item)
        yield tuple(window)

print(list(sliding_window([1, 2, 3, 4, 5], 3)))
print(list(sliding_window("ABCDE", 2)))

assert list(sliding_window([1, 2, 3, 4], 2)) == [(1, 2), (2, 3), (3, 4)]
assert list(sliding_window([1], 2)) == []
assert list(sliding_window([1, 2], 2)) == [(1, 2)]


[(1, 2, 3), (2, 3, 4), (3, 4, 5)]
[('A', 'B'), ('B', 'C'), ('C', 'D'), ('D', 'E')]


## Extra Challenge Solution 3 — `round_robin`


In [26]:
def round_robin(*iterables: Iterable):
    iterators = deque(iter(it) for it in iterables)

    while iterators:
        iterator = iterators.popleft()
        try:
            value = next(iterator)
        except StopIteration:
            continue
        else:
            yield value
            iterators.append(iterator)

print(list(round_robin("ABC", [1, 2], (True, False, None, "done"))))

assert list(round_robin("AB", [1, 2, 3])) == ["A", 1, "B", 2, 3]
assert list(round_robin([], [1], [], [2, 3])) == [1, 2, 3]
assert list(round_robin()) == []


['A', 1, True, 'B', 2, False, 'C', None, 'done']


# Final self-check

Run this cell after running the notebook from top to bottom. It verifies that the main custom tools still behave correctly.


In [27]:
assert list(SquaresIterator(5)) == [0, 1, 4, 9, 16]
assert list(Squares(5)) == [0, 1, 4, 9, 16]
assert list(Squares(5)) == [0, 1, 4, 9, 16]
assert SquareSequence(5)[-1] == 16
assert list(RandomNumbers(3, seed=123)) == list(RandomNumbers(3, seed=123))
assert list(islice(ArithmeticProgression(2, 2), 4)) == [2, 4, 6, 8]
assert list(triangular_numbers(4)) == [1, 3, 6, 10]
assert take(range(100), 3) == [0, 1, 2]
assert list(batched(range(5), 2)) == [(0, 1), (2, 3), (4,)]
assert list(flatten([1, [2, [3]], "ab"])) == [1, 2, 3, "ab"]
assert list(unique_everseen([1, 1, 2, 1, 3])) == [1, 2, 3]
assert list(my_enumerate(["x", "y"], 7)) == [(7, "x"), (8, "y")]
assert list(my_zip([1, 2], "abc")) == [(1, "a"), (2, "b")]
assert count_error_codes(only_errors(parse_log_lines(lines))) == {"E42": 2, "E99": 1}
assert list(sliding_window([1, 2, 3], 2)) == [(1, 2), (2, 3)]
assert list(round_robin("AB", [1, 2, 3])) == ["A", 1, "B", 2, 3]

print("All self-checks passed.")


All self-checks passed.


# Summary

The most important design decision is whether your object should be:

1. a **one-shot iterator**: `iter(obj) is obj`, state is stored on the object, and exhaustion is permanent; or
2. a **reusable iterable**: `iter(obj)` returns a fresh iterator each time.

Use one-shot iterators for streams and cursors. Use reusable iterables for collections.
